# Module 8b: exercises

Each exercise has you compute a test **by hand** from its formula; the check compares your answer with SciPy/statsmodels. Data: `app_trial.csv` (a randomised app experiment) and `students.csv`.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**One-sample $t$-test by hand.** Write `one_sample_t(x, mu0, level=0.95)` returning a dict with keys `t`, `df`, `p` (two-sided), `ci_low`, `ci_high`
(use $t=(\bar x-\mu_0)/(s/\sqrt n)$ and `scipy.stats.t` for the tail probability and critical value). Apply it to the exam scores against a benchmark of 65 as `res1`.

In [ ]:
import math

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import confint_proportions_2indep, proportions_ztest

trial = pd.read_csv(DATA / "app_trial.csv")
students = pd.read_csv(DATA / "students.csv")
by = {g: trial[trial["group"] == g] for g in ("control", "tutorial", "gamified")}

def one_sample_t(x, mu0, level=0.95):
    # YOUR CODE HERE: return {"t": ..., "df": ..., "p": ..., "ci_low": ..., "ci_high": ...}
    pass

res1 = None   # YOUR CODE HERE

In [ ]:
probe = np.random.default_rng(0).normal(10, 2, 25)
ref = stats.ttest_1samp(probe, 9.0)
mine = one_sample_t(probe, 9.0)
assert abs(mine["t"] - ref.statistic) < 1e-9 and abs(mine["p"] - ref.pvalue) < 1e-9 and mine["df"] == 24
ci = ref.confidence_interval(0.95)
assert abs(mine["ci_low"] - ci.low) < 1e-9 and abs(mine["ci_high"] - ci.high) < 1e-9
assert abs(res1["t"] - 3.953) < 0.01 and res1["p"] < 0.001 and res1["ci_low"] > 65
print(f"✅ Correct! t({res1['df']}) = {res1['t']:.2f}, p = {res1['p']:.5f}; the CI ({res1['ci_low']:.1f}, {res1['ci_high']:.1f}) excludes 65")

## Exercise 2

**Paired $t$-test.** Write `paired_t(before, after)` returning `{"mean_diff", "t", "df", "p", "dz"}`, computed from the within-person differences $d_i=\text{after}_i-\text{before}_i$
($t=\bar d/(s_d/\sqrt n)$, $d_z=\bar d/s_d$). Apply it to the **tutorial** group's `pre_score` and `post_score` as `res2`, and store the p-value of the *wrong* independent-samples Welch test on the same two columns as `p_wrong`.

In [ ]:
def paired_t(before, after):
    # YOUR CODE HERE
    pass

tut = by["tutorial"]
res2 = None      # YOUR CODE HERE
p_wrong = None   # YOUR CODE HERE

In [ ]:
ref = stats.ttest_rel(tut["post_score"], tut["pre_score"])
assert abs(res2["t"] - ref.statistic) < 1e-9 and abs(res2["p"] - ref.pvalue) < 1e-12 and res2["df"] == 99
assert abs(res2["mean_diff"] - 4.872) < 0.001 and abs(res2["dz"] - 0.759) < 0.001
assert abs(p_wrong - 0.0024) < 0.0005 and p_wrong > 1e6 * res2["p"], "The paired test is vastly more powerful here"
print(f"✅ Correct! Paired p = {res2['p']:.1e} vs {p_wrong:.4f} when the pairing is (wrongly) ignored")

## Exercise 3

**Mann–Whitney by ranks.** Write `mann_whitney_u(a, b)` returning `(U1, prob_superiority)`: rank all values together (`scipy.stats.rankdata` handles ties), then
$U_1=R_1-n_1(n_1+1)/2$ and the probability of superiority $U_1/(n_1n_2)$. Apply it to `minutes_active` for gamified (`a`) vs control (`b`) as `U1, prob`.

In [ ]:
def mann_whitney_u(a, b):
    # YOUR CODE HERE
    pass

mg = by["gamified"]["minutes_active"].to_numpy()
mc = by["control"]["minutes_active"].to_numpy()
U1, prob = None, None   # YOUR CODE HERE

In [ ]:
u, ps = mann_whitney_u([1, 2, 3], [4, 5, 6])
assert u == 0 and ps == 0, "every value in a is below every value in b"
u, ps = mann_whitney_u([4, 5, 6], [1, 2, 3])
assert u == 9 and ps == 1
assert U1 == stats.mannwhitneyu(mg, mc).statistic and abs(prob - 0.6231) < 1e-4
print(f"✅ Correct! U = {U1:.0f}; a random gamified user is more active than a random control user {prob:.1%} of the time")

## Exercise 4

**One-way ANOVA by hand.** Write `anova_f(*groups)` returning `(F, p, eta_squared)` from the between- and within-group sums of squares
($F=\frac{SSB/(k-1)}{SSW/(N-k)}$, $\eta^2=\frac{SSB}{SSB+SSW}$; use `stats.f.sf` for the p-value). Apply it to the three groups' `post_score` as `F, p, eta2`.

In [ ]:
def anova_f(*groups):
    # YOUR CODE HERE
    pass

post = [by[g]["post_score"].to_numpy() for g in ("control", "tutorial", "gamified")]
F, p, eta2 = None, None, None   # YOUR CODE HERE

In [ ]:
probe = [np.random.default_rng(i).normal(i, 1, 20) for i in range(3)]
ref = stats.f_oneway(*probe)
f0, p0, e0 = anova_f(*probe)
assert abs(f0 - ref.statistic) < 1e-9 and abs(p0 - ref.pvalue) < 1e-12 and 0 < e0 < 1
assert abs(F - 2.480) < 0.01 and abs(p - 0.0855) < 0.001 and abs(eta2 - 0.0164) < 0.001
print(f"✅ Correct! F(2, 297) = {F:.2f}, p = {p:.3f}, eta^2 = {eta2:.3f}: groups explain only {eta2:.1%} of the variance in post scores")

## Exercise 5

**Holm's correction.** Write `holm_adjust(pvals)` returning Holm-adjusted p-values in the *original order*. Method: sort the $m$ p-values ascending; the $j$-th smallest ($j=1..m$) is multiplied by $(m-j+1)$,
then take a running maximum (so adjusted values never decrease) and cap at 1. Apply it to `[0.01, 0.04, 0.03, 0.20]` as `adjusted`.

In [ ]:
def holm_adjust(pvals):
    # YOUR CODE HERE
    pass

raw = [0.01, 0.04, 0.03, 0.20]
adjusted = None   # YOUR CODE HERE

In [ ]:
assert np.allclose(adjusted, [0.04, 0.09, 0.09, 0.20]), adjusted
probe = np.random.default_rng(3).uniform(0, 0.1, 8)
assert np.allclose(holm_adjust(probe), multipletests(probe, method="holm")[1])
print("✅ Correct!", adjusted.round(2))

## Exercise 6

**Two proportions.** Write `two_prop_ztest(x1, n1, x2, n2)` returning a dict with `diff` ($\hat p_1-\hat p_2$), `z` (using the **pooled** proportion in the standard error), two-sided `p`, a 95% CI for the difference with the
**unpooled** standard error (`ci_low`, `ci_high`), and `odds_ratio`. Apply it to conversions: gamified (`x1`) vs control (`x2`) as `res6`.

In [ ]:
def two_prop_ztest(x1, n1, x2, n2):
    # YOUR CODE HERE
    pass

x1, n1 = int(by["gamified"]["converted"].sum()), len(by["gamified"])
x2, n2 = int(by["control"]["converted"].sum()), len(by["control"])
res6 = None   # YOUR CODE HERE

In [ ]:
z_ref, p_ref = proportions_ztest([x1, x2], [n1, n2])
lo, hi = confint_proportions_2indep(x1, n1, x2, n2, method="wald")
assert abs(res6["z"] - z_ref) < 1e-9 and abs(res6["p"] - p_ref) < 1e-9
assert abs(res6["ci_low"] - lo) < 0.002 and abs(res6["ci_high"] - hi) < 0.002, (res6["ci_low"], res6["ci_high"], lo, hi)
assert abs(res6["diff"] - 0.16) < 1e-9 and abs(res6["odds_ratio"] - 2.405) < 0.01
print(f"✅ Correct! difference {res6['diff']:.2f} (95% CI {res6['ci_low']:.2f} to {res6['ci_high']:.2f}), z = {res6['z']:.2f}, p = {res6['p']:.4f}, OR = {res6['odds_ratio']:.2f}")

## Exercise 7

**Chi-square test of independence by hand.** Write `chi2_independence(table)` for a 2-D array of counts, returning `{"chi2", "dof", "p", "cramers_v", "min_expected"}`
(expected counts $E_{ij}=\text{row}_i\times\text{col}_j/N$; $V=\sqrt{\chi^2/(N(\min(r,c)-1))}$). Apply it to `pd.crosstab(trial["group"], trial["device"])` as `res7`.

In [ ]:
def chi2_independence(table):
    # YOUR CODE HERE
    pass

res7 = None   # YOUR CODE HERE

In [ ]:
tab = pd.crosstab(trial["group"], trial["device"]).to_numpy()
ref = stats.chi2_contingency(tab, correction=False)
assert abs(res7["chi2"] - ref[0]) < 1e-9 and abs(res7["p"] - ref[1]) < 1e-9 and res7["dof"] == ref[2] == 4
assert abs(res7["min_expected"] - 14.667) < 0.01 and abs(res7["cramers_v"] - 0.0552) < 0.001
print(f"✅ Correct! chi2(4) = {res7['chi2']:.2f}, p = {res7['p']:.2f}, V = {res7['cramers_v']:.3f}: no evidence that devices were unevenly spread across groups")

## Exercise 8

**A confidence interval for a correlation.** Write `corr_ci(r, n, level=0.95)` using Fisher's transformation: $z=\operatorname{arctanh}(r)$, standard error $1/\sqrt{n-3}$, a normal interval for $z$ using `stats.norm.ppf`,
then back-transform with `tanh`. Apply it to the pre/post correlation in `trial` as `ci_pre_post`.

In [ ]:
def corr_ci(r, n, level=0.95):
    # YOUR CODE HERE
    pass

r_pre_post = trial["pre_score"].corr(trial["post_score"])
ci_pre_post = None   # YOUR CODE HERE

In [ ]:
lo, hi = corr_ci(0.5, 28)
assert abs(lo - 0.1560) < 1e-3 and abs(hi - 0.7358) < 1e-3, (lo, hi)
assert abs(ci_pre_post[0] - 0.818) < 0.002 and abs(ci_pre_post[1] - 0.881) < 0.002, ci_pre_post
assert ci_pre_post[0] < r_pre_post < ci_pre_post[1]
print(f"✅ Correct! r = {r_pre_post:.3f}, 95% CI ({ci_pre_post[0]:.3f}, {ci_pre_post[1]:.3f})")

## Exercise 9

**Plan the sample size.** With `TTestIndPower` (two-sample $t$-test, $\alpha=0.05$), compute `n_per_group` (an integer, rounded **up**) needed to detect $d=0.5$ with 80% power;
`power_n100` the power to detect $d=0.3$ with 100 per group; and `mde` the smallest effect size detectable with 80% power at 100 per group.

In [ ]:
power = TTestIndPower()
n_per_group = power_n100 = mde = None   # YOUR CODE HERE

In [ ]:
assert n_per_group == 64, n_per_group
assert abs(power_n100 - 0.56) < 0.01, power_n100
assert abs(mde - 0.398) < 0.005, mde
print(f"✅ Correct! Need {n_per_group} per group for d = 0.5; with 100 per group you'd detect d >= {mde:.2f} and miss d = 0.3 about {1 - power_n100:.0%} of the time")

## Exercise 10

**Choose the test.** For each situation set the value in `answers` to one of: `"welch_t"`, `"paired_t"`, `"mann_whitney"`, `"anova"`, `"two_proportion_z"`, `"chi_square_independence"`, `"fisher_exact"`, `"pearson"`.

* **a)** Compare mean exam scores between two different classes (roughly normal, similar spreads).
* **b)** The same 30 patients' blood pressure before and after a treatment (differences roughly normal).
* **c)** Compare time-on-task between two groups; times are extremely right-skewed with outliers.
* **d)** Compare mean satisfaction across four independent store locations (roughly normal, equal spreads).
* **e)** Compare the share of customers who buy, in two independent groups of 500.
* **f)** Test whether favourite colour is related to region (a 4×5 table with large counts).
* **g)** A 2×2 table of outcomes with only 12 people total (expected counts of 2–4).

In [ ]:
answers = {"a": None, "b": None, "c": None, "d": None, "e": None, "f": None, "g": None}   # YOUR CODE HERE

In [ ]:
expected = {"a": "welch_t", "b": "paired_t", "c": "mann_whitney", "d": "anova", "e": "two_proportion_z", "f": "chi_square_independence", "g": "fisher_exact"}
for key, want in expected.items():
    assert answers[key] == want, f"({key}) re-read the decision table: what is the design and what type is the outcome?"
print("✅ Correct!")